<a href="https://colab.research.google.com/github/SiyumiJayawardhane/sinhala-wsd/blob/extraction-pipeline/Extraction_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Notebook setup - drive, HF login, paths, seed

In [4]:
!pip install -q datasets huggingface_hub datasketch openpyxl

from google.colab import drive, userdata
from huggingface_hub import login
import os

drive.mount('/content/drive')
login(token=userdata.get('HF_TOKEN'))

BASE        = "/content/drive/MyDrive/Research-WSD-New"
OUT         = f"{BASE}/extraction_v3"
os.makedirs(OUT, exist_ok=True)

RANDOM_SEED = 42
RESERVOIR_K = 1000   # sentences kept per word per source during the scan

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


2. Load target words

In [6]:
import re, itertools, unicodedata
from collections import defaultdict
import openpyxl

WORDS_XLSX = "/content/drive/MyDrive/Research-WSD-New/sinhala_polysemy_words.xlsx"
REMOVED    = {"වේ"}

def load_target_words(path=WORDS_XLSX, sheet="words"):
    """Reads No | Word | forms. Returns the word list (in sheet order), {word: No},
    {word: [spellings]} and {word: [forms]}. 'පස / පස්' stays ONE word."""
    ws = openpyxl.load_workbook(path)[sheet]
    rows = ws.iter_rows(values_only=True)
    header = [str(h).strip() if h else "" for h in next(rows)]          # 'No ' has a trailing space
    i_no, i_word = header.index("No"), header.index("Word")
    # the forms column is called "forms (rule-based)" in the sheet; "Word forms" also accepted
    i_forms = next(i for i, h in enumerate(header) if h in ("Word forms", "forms (rule-based)"))
    words, numbers, spellings, forms = [], {}, {}, {}
    for r in rows:
        r = tuple(r) + (None,) * len(header)
        if not r[i_word]:
            continue
        w = str(r[i_word]).strip()
        sp = [s.strip() for s in re.split(r"[/,]", w) if s.strip()]    # "පස / පස්" -> ["පස", "පස්"]
        if w in REMOVED or any(s in REMOVED for s in sp):
            continue
        if w in numbers:
            raise ValueError(f"Duplicate word in sheet: {w}")
        f = [x.strip() for x in str(r[i_forms] or "").split(",") if x.strip()]
        if not f:
            raise ValueError(f"No forms for {w}")
        words.append(w)
        numbers[w] = int(float(r[i_no]))
        spellings[w] = sp
        forms[w] = list(dict.fromkeys(sp + f))
    return words, numbers, spellings, forms

target_words, word_no, word_spellings, word_forms = load_target_words()
print(f"Loaded {len(target_words)} target words, {sum(map(len, word_forms.values())):,} forms")
print(target_words)
assert len(target_words) == 67, f"Expected 67 words, got {len(target_words)}"

# never change the sense, so a token = form + up to 2 clitics. -උත් is written as the vowel sign ුත්.
CLITICS = ["ත්", "ුත්", "ද", "ම", "යි", "කි", "වත්", "ය"]
ENDINGS = [""] + CLITICS + ["".join(p) for p in itertools.product(CLITICS, repeat=2)]

JOINERS = re.compile(r"[\u200C\u200D]")
def match_key(tok):
    """Key used ONLY for matching: NFC + ZWJ/ZWNJ removed, so ප්\u200Dර and ප්ර match the same."""
    return JOINERS.sub("", unicodedata.normalize("NFC", tok))

# WITH word forms: token key -> words / base form
KEY2WORDS = defaultdict(set)
KEY2FORM  = {}
for w in target_words:
    for f in word_forms[w]:
        for e in ENDINGS:
            k = match_key(f + e)
            KEY2WORDS[k].add(w)
            KEY2FORM.setdefault((w, k), f)

# WITHOUT word forms (target word only): just the word's own spelling(s), exact whole token
BASE_KEY2WORD = {match_key(s): w for w in target_words for s in word_spellings[w]}

# same matchers in the shape the extraction helpers (cell 3) use
MATCHERS = {"forms": defaultdict(dict), "base": defaultdict(dict)}
for (w, k), f in KEY2FORM.items():
    MATCHERS["forms"][k][w] = f
for w in target_words:
    for s in word_spellings[w]:
        MATCHERS["base"][match_key(s)][w] = s

TARGETS = target_words
shared = {k: sorted(v) for k, v in KEY2WORDS.items() if len(v) > 1}
print(f"{len(KEY2WORDS):,} token keys | {len(shared)} shared by 2+ words | {len(BASE_KEY2WORD)} target-word spellings")

Loaded 67 target words, 1,370 forms
['දිය', 'දද', 'නා', 'ගත', 'පස / පස්', 'මුව', 'වත', 'අඹ', 'තෙර', 'දන', 'ලිය', 'වණ', 'උර', 'තල', 'නල', 'පෙළ', 'මද', 'මී', 'රැස', 'වර', 'වැට', 'අත', 'ගල', 'උත්තර', 'කඩ', 'කම්', 'කලාපය', 'සුර', 'බල', 'නද', 'සත්', 'සල් / සල', 'පත්', 'සක', 'පැහ, පැහැ', 'තුරු', 'තිර', 'සඳ', 'සිළු', 'නුඹ', 'හස', 'නිදන', 'පණ', 'පිය', 'දත්', 'කලි', 'පට', 'රද', 'දෙස්', 'සමර', 'වස්', 'සර', 'කුළු', 'සිරි', 'කර', 'කමල් / කමල', 'කන', 'රජ', 'රස', 'රා', 'වැඩ', 'දල', 'ලේ', 'සී', 'වන', 'ලද', 'කල/කල්']
99,724 token keys | 0 shared by 2+ words | 72 target-word spellings
